# PlatformPulse — incident postmortem
Analysis of the eval run (`eval/results.csv`) and the incident service history.

Run with the stack up: `pip install pandas matplotlib requests` then open in Jupyter.

In [ ]:
import pandas as pd, requests, matplotlib.pyplot as plt
ev = pd.read_csv('../eval/results.csv')
ev.head()

## Triage accuracy by failure type

In [ ]:
acc = ev.groupby('expected')['correct'].mean().rename('accuracy')
acc.plot.bar(ylim=(0, 1), title='Agent triage accuracy'); acc

## Where the time goes: detection vs agent triage

In [ ]:
ev.groupby('expected')[['detect_s', 'triage_s']].median().plot.bar(stacked=True, title='Median seconds: chaos -> incident -> summary')
ev[['detect_s', 'triage_s', 'total_s']].describe()

## Misclassifications

In [ ]:
ev[~ev['correct'].astype(bool)][['run', 'expected', 'predicted', 'tools_used']]

## Incident history (live from the incident service)

In [ ]:
inc = pd.DataFrame(requests.get('http://localhost:8000/incidents', params={'limit': 200}).json())
inc['mtta_s'] = inc['acked_at'] - inc['created_at']
inc['mttr_s'] = inc['resolved_at'] - inc['created_at']
inc.groupby(['alertname', 'category'], dropna=False).agg(n=('id', 'count'), mtta_s=('mtta_s', 'mean'), mttr_s=('mttr_s', 'mean'))

In [ ]:
inc['resolved_by'].value_counts(dropna=False).plot.pie(autopct='%.0f%%', title='Resolved by (deflection)')

## Takeaways
- Fill in after reviewing the numbers above: which type is slowest to detect, where the agent errs, and what to fix next.